"""
SBTi Progression Tracker
Analyzes how companies progress from near-term commitments to net zero targets
"""

import pandas as pd
import numpy as np
from collections import defaultdict

# Read data
df = pd.read_excel(
    "historic new.xlsx",
    sheet_name="historic new"  
)
df.columns = df.columns.str.strip().str.replace('\ufeff', '')

print("="*80)
print("SBTi PROGRESSION TRACKER")
print("="*80)

# Create progression tracking
progressions = []

for idx, row in df.iterrows():
    company = row['company']
    
    # Extract statuses for each year
    years = ['2021', '2022', '2023', '2024']
    nt_status = {year: row[f'{year}_NT_Status'] for year in years}
    nz_status = {year: row[f'{year}_NZ_Status'] for year in years}
    
    # Clean NaN values
    for year in years:
        if pd.isna(nt_status[year]):
            nt_status[year] = 'None'
        if pd.isna(nz_status[year]):
            nz_status[year] = 'None'
    
    # Create progression string
    nt_progression = ' → '.join([f"{year}:{nt_status[year]}" for year in years])
    nz_progression = ' → '.join([f"{year}:{nz_status[year]}" for year in years])
    
    # Determine pathway type
    pathway_type = "No commitment"
    
    # Check if NT commitment led to NZ commitment
    has_nt_any_year = any(nt_status[year] in ['C', 'T'] for year in years)
    has_nz_any_year = any(nz_status[year] in ['C', 'T'] for year in years)
    
    if has_nt_any_year and has_nz_any_year:
        # Check timing
        first_nt_year = None
        first_nz_year = None
        
        for year in years:
            if nt_status[year] in ['C', 'T'] and first_nt_year is None:
                first_nt_year = year
            if nz_status[year] in ['C', 'T'] and first_nz_year is None:
                first_nz_year = year
        
        if first_nt_year and first_nz_year:
            nt_year_num = int(first_nt_year)
            nz_year_num = int(first_nz_year)
            
            if nt_year_num <= nz_year_num:
                pathway_type = "NT → NZ pathway"
            elif nz_year_num < nt_year_num:
                pathway_type = "NZ before NT"
            else:
                pathway_type = "NT and NZ simultaneous"
    elif has_nt_any_year and not has_nz_any_year:
        pathway_type = "NT only"
    elif not has_nt_any_year and has_nz_any_year:
        pathway_type = "NZ only (no NT)"
    
    # Current status (2024)
    current_nt = nt_status['2024']
    current_nz = nz_status['2024']
    
    progressions.append({
        'Company': company,
        'Pathway_Type': pathway_type,
        'Current_NT_2024': current_nt,
        'Current_NZ_2024': current_nz,
        'NT_Progression': nt_progression,
        'NZ_Progression': nz_progression,
        'NT_2021': nt_status['2021'],
        'NT_2022': nt_status['2022'],
        'NT_2023': nt_status['2023'],
        'NT_2024': nt_status['2024'],
        'NZ_2021': nz_status['2021'],
        'NZ_2022': nz_status['2022'],
        'NZ_2023': nz_status['2023'],
        'NZ_2024': nz_status['2024'],
    })

progression_df = pd.DataFrame(progressions)

# Save full progression data
progression_df.to_csv('/mnt/user-data/outputs/company_sbti_progressions.csv', index=False)
print("\n✅ Saved full progression data to: company_sbti_progressions.csv")

# ============================================================================
# ANALYSIS 1: Pathway Type Distribution
# ============================================================================
print("\n" + "="*80)
print("ANALYSIS 1: PATHWAY TYPES")
print("="*80)

pathway_counts = progression_df['Pathway_Type'].value_counts()
print("\n📊 Distribution of pathway types:")
for pathway, count in pathway_counts.items():
    pct = (count / len(progression_df)) * 100
    print(f"   {pathway:.<40} {count:>4} ({pct:>5.1f}%)")

# ============================================================================
# ANALYSIS 2: NT Commitment → NZ Commitment Flow
# ============================================================================
print("\n" + "="*80)
print("ANALYSIS 2: NEAR-TERM → NET ZERO PATHWAY")
print("="*80)

nt_to_nz = progression_df[progression_df['Pathway_Type'] == 'NT → NZ pathway']
print(f"\n✅ Companies that followed NT → NZ pathway: {len(nt_to_nz)}")

if len(nt_to_nz) > 0:
    print("\n🔍 Examples of NT → NZ progression:")
    for idx, row in nt_to_nz.head(10).iterrows():
        print(f"\n   {row['Company']}")
        print(f"      NT: {row['NT_Progression']}")
        print(f"      NZ: {row['NZ_Progression']}")

# ============================================================================
# ANALYSIS 3: NT Committed but No NZ
# ============================================================================
print("\n" + "="*80)
print("ANALYSIS 3: COMPANIES WITH NT BUT NO NET ZERO")
print("="*80)

nt_only = progression_df[progression_df['Pathway_Type'] == 'NT only']
print(f"\n📌 Companies with near-term targets but no net zero commitment: {len(nt_only)}")

# Break down by current NT status
nt_only_by_status = nt_only['Current_NT_2024'].value_counts()
print("\n   Current status breakdown:")
for status, count in nt_only_by_status.items():
    print(f"      {status}: {count} companies")

# ============================================================================
# ANALYSIS 4: Timing Analysis - How long from NT to NZ?
# ============================================================================
print("\n" + "="*80)
print("ANALYSIS 4: TIMING - HOW LONG FROM NT TO NZ?")
print("="*80)

timing_analysis = []

for idx, row in nt_to_nz.iterrows():
    # Find first year with NT
    first_nt = None
    for year in ['2021', '2022', '2023', '2024']:
        if row[f'NT_{year}'] in ['C', 'T']:
            first_nt = int(year)
            break
    
    # Find first year with NZ
    first_nz = None
    for year in ['2021', '2022', '2023', '2024']:
        if row[f'NZ_{year}'] in ['C', 'T']:
            first_nz = int(year)
            break
    
    if first_nt and first_nz:
        time_diff = first_nz - first_nt
        timing_analysis.append({
            'Company': row['Company'],
            'First_NT_Year': first_nt,
            'First_NZ_Year': first_nz,
            'Years_Between': time_diff,
            'Same_Year': time_diff == 0
        })

timing_df = pd.DataFrame(timing_analysis)

if len(timing_df) > 0:
    same_year = timing_df['Same_Year'].sum()
    later = len(timing_df) - same_year
    
    print(f"\n⏱️ Timing breakdown:")
    print(f"   Set NT and NZ simultaneously: {same_year} companies ({same_year/len(timing_df)*100:.1f}%)")
    print(f"   Set NZ after NT: {later} companies ({later/len(timing_df)*100:.1f}%)")
    
    if later > 0:
        avg_time = timing_df[timing_df['Years_Between'] > 0]['Years_Between'].mean()
        print(f"   Average time between NT and NZ: {avg_time:.1f} years")

# ============================================================================
# ANALYSIS 5: Status Combinations in 2024
# ============================================================================
print("\n" + "="*80)
print("ANALYSIS 5: CURRENT STATUS COMBINATIONS (2024)")
print("="*80)

status_combos = progression_df.groupby(['Current_NT_2024', 'Current_NZ_2024']).size().reset_index(name='Count')
status_combos = status_combos.sort_values('Count', ascending=False)

print("\n📊 NT Status x NZ Status combinations:")
print("\n   NT Status  |  NZ Status  |  Count")
print("   " + "-"*40)
for _, row in status_combos.iterrows():
    print(f"   {row['Current_NT_2024']:10} | {row['Current_NZ_2024']:11} | {row['Count']:>5}")

# ============================================================================
# ANALYSIS 6: Evolution Patterns
# ============================================================================
print("\n" + "="*80)
print("ANALYSIS 6: SPECIFIC EVOLUTION PATTERNS")
print("="*80)

# Pattern 1: NT Committed → NT Targets Set → NZ Committed
pattern1 = []
for idx, row in progression_df.iterrows():
    # Check if progression goes: NT commit → NT target → NZ commit
    has_pattern = False
    
    # Look for NT commitment first
    nt_commit_year = None
    for year in ['2021', '2022', '2023', '2024']:
        if row[f'NT_{year}'] == 'C' and nt_commit_year is None:
            nt_commit_year = year
    
    # Then NT targets set
    nt_target_year = None
    for year in ['2021', '2022', '2023', '2024']:
        if row[f'NT_{year}'] == 'T' and nt_target_year is None:
            if nt_commit_year is None or int(year) >= int(nt_commit_year):
                nt_target_year = year
    
    # Then NZ commitment
    nz_year = None
    for year in ['2021', '2022', '2023', '2024']:
        if row[f'NZ_{year}'] in ['C', 'T'] and nz_year is None:
            nz_year = year
    
    if nt_commit_year and nt_target_year and nz_year:
        pattern1.append({
            'Company': row['Company'],
            'Pattern': f'NT_Commit({nt_commit_year}) → NT_Target({nt_target_year}) → NZ({nz_year})'
        })

print(f"\n🎯 Pattern 1: NT Committed → NT Targets Set → NZ")
print(f"   Found: {len(pattern1)} companies")
if len(pattern1) > 0:
    print("\n   Examples:")
    for item in pattern1[:5]:
        print(f"      {item['Company']}: {item['Pattern']}")

# Pattern 2: NT and NZ together from start
pattern2 = progression_df[
    (progression_df['NT_2021'].isin(['C', 'T'])) & 
    (progression_df['NZ_2021'].isin(['C', 'T']))
]

print(f"\n🎯 Pattern 2: NT and NZ committed together from 2021")
print(f"   Found: {len(pattern2)} companies")
if len(pattern2) > 0:
    print("\n   Examples:")
    for idx, row in pattern2.head(5).iterrows():
        print(f"      {row['Company']}")

# ============================================================================
# ANALYSIS 7: Companies that DROPPED NZ after having it
# ============================================================================
print("\n" + "="*80)
print("ANALYSIS 7: COMPANIES THAT DROPPED NET ZERO")
print("="*80)

dropped_nz = []
for idx, row in progression_df.iterrows():
    had_nz = False
    lost_nz = False
    
    for year in ['2021', '2022', '2023']:
        if row[f'NZ_{year}'] in ['C', 'T']:
            had_nz = True
            break
    
    if had_nz and row['NZ_2024'] == 'None':
        lost_nz = True
    
    if lost_nz:
        dropped_nz.append({
            'Company': row['Company'],
            'NZ_2021': row['NZ_2021'],
            'NZ_2022': row['NZ_2022'],
            'NZ_2023': row['NZ_2023'],
            'NZ_2024': row['NZ_2024'],
            'NT_2024': row['NT_2024']
        })

dropped_df = pd.DataFrame(dropped_nz)
print(f"\n⚠️ Companies that had NZ commitment but lost it: {len(dropped_df)}")

if len(dropped_df) > 0:
    print("\n   Top cases:")
    for idx, row in dropped_df.head(10).iterrows():
        print(f"\n      {row['Company']}")
        print(f"         NZ: 2021={row['NZ_2021']} → 2022={row['NZ_2022']} → 2023={row['NZ_2023']} → 2024={row['NZ_2024']}")
        print(f"         NT in 2024: {row['NT_2024']}")

# ============================================================================
# SUMMARY STATISTICS
# ============================================================================
print("\n" + "="*80)
print("SUMMARY STATISTICS")
print("="*80)

total_companies = len(progression_df)
has_nt_2024 = len(progression_df[progression_df['Current_NT_2024'].isin(['C', 'T'])])
has_nz_2024 = len(progression_df[progression_df['Current_NZ_2024'].isin(['C', 'T'])])
has_both_2024 = len(progression_df[
    (progression_df['Current_NT_2024'].isin(['C', 'T'])) & 
    (progression_df['Current_NZ_2024'].isin(['C', 'T']))
])

print(f"""
📊 2024 Status:
   Total companies tracked: {total_companies}
   
   Have Near-term SBTi: {has_nt_2024} ({has_nt_2024/total_companies*100:.1f}%)
   Have Net Zero commitment: {has_nz_2024} ({has_nz_2024/total_companies*100:.1f}%)
   Have BOTH NT and NZ: {has_both_2024} ({has_both_2024/total_companies*100:.1f}%)
   
   Have NT but NO NZ: {has_nt_2024 - has_both_2024} ({(has_nt_2024-has_both_2024)/total_companies*100:.1f}%)
   Have NZ but NO NT: {has_nz_2024 - has_both_2024} ({(has_nz_2024-has_both_2024)/total_companies*100:.1f}%)
""")

# Calculate conversion rate: companies with NT that also have NZ
if has_nt_2024 > 0:
    conversion_rate = (has_both_2024 / has_nt_2024) * 100
    print(f"   🎯 Conversion Rate: {conversion_rate:.1f}% of companies with NT also have NZ")

print("\n" + "="*80)
print("✅ ANALYSIS COMPLETE")
print("="*80)
print(f"\n📁 Output files saved to /mnt/user-data/outputs/")
print(f"   1. company_sbti_progressions.csv - Full progression data for all companies")

In [2]:
"""
SBTi Progression Tracker
Analyzes how companies progress from near-term commitments to net zero targets
"""

import pandas as pd
import numpy as np
from collections import defaultdict

# Read data
df = pd.read_excel(
    "historic new.xlsx",
    sheet_name="historic new"  
)
df.columns = df.columns.str.strip().str.replace('\ufeff', '')

print("="*80)
print("SBTi PROGRESSION TRACKER")
print("="*80)

# Create progression tracking
progressions = []

for idx, row in df.iterrows():
    company = row['company']
    
    # Extract statuses for each year
    years = ['2021', '2022', '2023', '2024']
    nt_status = {year: row[f'{year}_NT_Status'] for year in years}
    nz_status = {year: row[f'{year}_NZ_Status'] for year in years}
    
    # Clean NaN values
    for year in years:
        if pd.isna(nt_status[year]):
            nt_status[year] = 'None'
        if pd.isna(nz_status[year]):
            nz_status[year] = 'None'
    
    # Create progression string
    nt_progression = ' → '.join([f"{year}:{nt_status[year]}" for year in years])
    nz_progression = ' → '.join([f"{year}:{nz_status[year]}" for year in years])
    
    # Determine pathway type
    pathway_type = "No commitment"
    
    # Check if NT commitment led to NZ commitment
    has_nt_any_year = any(nt_status[year] in ['C', 'T'] for year in years)
    has_nz_any_year = any(nz_status[year] in ['C', 'T'] for year in years)
    
    if has_nt_any_year and has_nz_any_year:
        # Check timing
        first_nt_year = None
        first_nz_year = None
        
        for year in years:
            if nt_status[year] in ['C', 'T'] and first_nt_year is None:
                first_nt_year = year
            if nz_status[year] in ['C', 'T'] and first_nz_year is None:
                first_nz_year = year
        
        if first_nt_year and first_nz_year:
            nt_year_num = int(first_nt_year)
            nz_year_num = int(first_nz_year)
            
            if nt_year_num <= nz_year_num:
                pathway_type = "NT → NZ pathway"
            elif nz_year_num < nt_year_num:
                pathway_type = "NZ before NT"
            else:
                pathway_type = "NT and NZ simultaneous"
    elif has_nt_any_year and not has_nz_any_year:
        pathway_type = "NT only"
    elif not has_nt_any_year and has_nz_any_year:
        pathway_type = "NZ only (no NT)"
    
    # Current status (2024)
    current_nt = nt_status['2024']
    current_nz = nz_status['2024']
    
    progressions.append({
        'Company': company,
        'Pathway_Type': pathway_type,
        'Current_NT_2024': current_nt,
        'Current_NZ_2024': current_nz,
        'NT_Progression': nt_progression,
        'NZ_Progression': nz_progression,
        'NT_2021': nt_status['2021'],
        'NT_2022': nt_status['2022'],
        'NT_2023': nt_status['2023'],
        'NT_2024': nt_status['2024'],
        'NZ_2021': nz_status['2021'],
        'NZ_2022': nz_status['2022'],
        'NZ_2023': nz_status['2023'],
        'NZ_2024': nz_status['2024'],
    })

progression_df = pd.DataFrame(progressions)

# Save full progression data
progression_df.to_csv('/mnt/user-data/outputs/company_sbti_progressions.csv', index=False)
print("\n✅ Saved full progression data to: company_sbti_progressions.csv")

# ============================================================================
# ANALYSIS 1: Pathway Type Distribution
# ============================================================================
print("\n" + "="*80)
print("ANALYSIS 1: PATHWAY TYPES")
print("="*80)

pathway_counts = progression_df['Pathway_Type'].value_counts()
print("\n📊 Distribution of pathway types:")
for pathway, count in pathway_counts.items():
    pct = (count / len(progression_df)) * 100
    print(f"   {pathway:.<40} {count:>4} ({pct:>5.1f}%)")

# ============================================================================
# ANALYSIS 2: NT Commitment → NZ Commitment Flow
# ============================================================================
print("\n" + "="*80)
print("ANALYSIS 2: NEAR-TERM → NET ZERO PATHWAY")
print("="*80)

nt_to_nz = progression_df[progression_df['Pathway_Type'] == 'NT → NZ pathway']
print(f"\n✅ Companies that followed NT → NZ pathway: {len(nt_to_nz)}")

if len(nt_to_nz) > 0:
    print("\n🔍 Examples of NT → NZ progression:")
    for idx, row in nt_to_nz.head(10).iterrows():
        print(f"\n   {row['Company']}")
        print(f"      NT: {row['NT_Progression']}")
        print(f"      NZ: {row['NZ_Progression']}")

# ============================================================================
# ANALYSIS 3: NT Committed but No NZ
# ============================================================================
print("\n" + "="*80)
print("ANALYSIS 3: COMPANIES WITH NT BUT NO NET ZERO")
print("="*80)

nt_only = progression_df[progression_df['Pathway_Type'] == 'NT only']
print(f"\n📌 Companies with near-term targets but no net zero commitment: {len(nt_only)}")

# Break down by current NT status
nt_only_by_status = nt_only['Current_NT_2024'].value_counts()
print("\n   Current status breakdown:")
for status, count in nt_only_by_status.items():
    print(f"      {status}: {count} companies")

# ============================================================================
# ANALYSIS 4: Timing Analysis - How long from NT to NZ?
# ============================================================================
print("\n" + "="*80)
print("ANALYSIS 4: TIMING - HOW LONG FROM NT TO NZ?")
print("="*80)

timing_analysis = []

for idx, row in nt_to_nz.iterrows():
    # Find first year with NT
    first_nt = None
    for year in ['2021', '2022', '2023', '2024']:
        if row[f'NT_{year}'] in ['C', 'T']:
            first_nt = int(year)
            break
    
    # Find first year with NZ
    first_nz = None
    for year in ['2021', '2022', '2023', '2024']:
        if row[f'NZ_{year}'] in ['C', 'T']:
            first_nz = int(year)
            break
    
    if first_nt and first_nz:
        time_diff = first_nz - first_nt
        timing_analysis.append({
            'Company': row['Company'],
            'First_NT_Year': first_nt,
            'First_NZ_Year': first_nz,
            'Years_Between': time_diff,
            'Same_Year': time_diff == 0
        })

timing_df = pd.DataFrame(timing_analysis)

if len(timing_df) > 0:
    same_year = timing_df['Same_Year'].sum()
    later = len(timing_df) - same_year
    
    print(f"\n⏱️ Timing breakdown:")
    print(f"   Set NT and NZ simultaneously: {same_year} companies ({same_year/len(timing_df)*100:.1f}%)")
    print(f"   Set NZ after NT: {later} companies ({later/len(timing_df)*100:.1f}%)")
    
    if later > 0:
        avg_time = timing_df[timing_df['Years_Between'] > 0]['Years_Between'].mean()
        print(f"   Average time between NT and NZ: {avg_time:.1f} years")

# ============================================================================
# ANALYSIS 5: Status Combinations in 2024
# ============================================================================
print("\n" + "="*80)
print("ANALYSIS 5: CURRENT STATUS COMBINATIONS (2024)")
print("="*80)

status_combos = progression_df.groupby(['Current_NT_2024', 'Current_NZ_2024']).size().reset_index(name='Count')
status_combos = status_combos.sort_values('Count', ascending=False)

print("\n📊 NT Status x NZ Status combinations:")
print("\n   NT Status  |  NZ Status  |  Count")
print("   " + "-"*40)
for _, row in status_combos.iterrows():
    print(f"   {row['Current_NT_2024']:10} | {row['Current_NZ_2024']:11} | {row['Count']:>5}")

# ============================================================================
# ANALYSIS 6: Evolution Patterns
# ============================================================================
print("\n" + "="*80)
print("ANALYSIS 6: SPECIFIC EVOLUTION PATTERNS")
print("="*80)

# Pattern 1: NT Committed → NT Targets Set → NZ Committed
pattern1 = []
for idx, row in progression_df.iterrows():
    # Check if progression goes: NT commit → NT target → NZ commit
    has_pattern = False
    
    # Look for NT commitment first
    nt_commit_year = None
    for year in ['2021', '2022', '2023', '2024']:
        if row[f'NT_{year}'] == 'C' and nt_commit_year is None:
            nt_commit_year = year
    
    # Then NT targets set
    nt_target_year = None
    for year in ['2021', '2022', '2023', '2024']:
        if row[f'NT_{year}'] == 'T' and nt_target_year is None:
            if nt_commit_year is None or int(year) >= int(nt_commit_year):
                nt_target_year = year
    
    # Then NZ commitment
    nz_year = None
    for year in ['2021', '2022', '2023', '2024']:
        if row[f'NZ_{year}'] in ['C', 'T'] and nz_year is None:
            nz_year = year
    
    if nt_commit_year and nt_target_year and nz_year:
        pattern1.append({
            'Company': row['Company'],
            'Pattern': f'NT_Commit({nt_commit_year}) → NT_Target({nt_target_year}) → NZ({nz_year})'
        })

print(f"\n🎯 Pattern 1: NT Committed → NT Targets Set → NZ")
print(f"   Found: {len(pattern1)} companies")
if len(pattern1) > 0:
    print("\n   Examples:")
    for item in pattern1[:5]:
        print(f"      {item['Company']}: {item['Pattern']}")

# Pattern 2: NT and NZ together from start
pattern2 = progression_df[
    (progression_df['NT_2021'].isin(['C', 'T'])) & 
    (progression_df['NZ_2021'].isin(['C', 'T']))
]

print(f"\n🎯 Pattern 2: NT and NZ committed together from 2021")
print(f"   Found: {len(pattern2)} companies")
if len(pattern2) > 0:
    print("\n   Examples:")
    for idx, row in pattern2.head(5).iterrows():
        print(f"      {row['Company']}")

# ============================================================================
# ANALYSIS 7: Companies that DROPPED NZ after having it
# ============================================================================
print("\n" + "="*80)
print("ANALYSIS 7: COMPANIES THAT DROPPED NET ZERO")
print("="*80)

dropped_nz = []
for idx, row in progression_df.iterrows():
    had_nz = False
    lost_nz = False
    
    for year in ['2021', '2022', '2023']:
        if row[f'NZ_{year}'] in ['C', 'T']:
            had_nz = True
            break
    
    if had_nz and row['NZ_2024'] == 'None':
        lost_nz = True
    
    if lost_nz:
        dropped_nz.append({
            'Company': row['Company'],
            'NZ_2021': row['NZ_2021'],
            'NZ_2022': row['NZ_2022'],
            'NZ_2023': row['NZ_2023'],
            'NZ_2024': row['NZ_2024'],
            'NT_2024': row['NT_2024']
        })

dropped_df = pd.DataFrame(dropped_nz)
print(f"\n⚠️ Companies that had NZ commitment but lost it: {len(dropped_df)}")

if len(dropped_df) > 0:
    print("\n   Top cases:")
    for idx, row in dropped_df.head(10).iterrows():
        print(f"\n      {row['Company']}")
        print(f"         NZ: 2021={row['NZ_2021']} → 2022={row['NZ_2022']} → 2023={row['NZ_2023']} → 2024={row['NZ_2024']}")
        print(f"         NT in 2024: {row['NT_2024']}")

# ============================================================================
# SUMMARY STATISTICS
# ============================================================================
print("\n" + "="*80)
print("SUMMARY STATISTICS")
print("="*80)

total_companies = len(progression_df)
has_nt_2024 = len(progression_df[progression_df['Current_NT_2024'].isin(['C', 'T'])])
has_nz_2024 = len(progression_df[progression_df['Current_NZ_2024'].isin(['C', 'T'])])
has_both_2024 = len(progression_df[
    (progression_df['Current_NT_2024'].isin(['C', 'T'])) & 
    (progression_df['Current_NZ_2024'].isin(['C', 'T']))
])

print(f"""
📊 2024 Status:
   Total companies tracked: {total_companies}
   
   Have Near-term SBTi: {has_nt_2024} ({has_nt_2024/total_companies*100:.1f}%)
   Have Net Zero commitment: {has_nz_2024} ({has_nz_2024/total_companies*100:.1f}%)
   Have BOTH NT and NZ: {has_both_2024} ({has_both_2024/total_companies*100:.1f}%)
   
   Have NT but NO NZ: {has_nt_2024 - has_both_2024} ({(has_nt_2024-has_both_2024)/total_companies*100:.1f}%)
   Have NZ but NO NT: {has_nz_2024 - has_both_2024} ({(has_nz_2024-has_both_2024)/total_companies*100:.1f}%)
""")

# Calculate conversion rate: companies with NT that also have NZ
if has_nt_2024 > 0:
    conversion_rate = (has_both_2024 / has_nt_2024) * 100
    print(f"   🎯 Conversion Rate: {conversion_rate:.1f}% of companies with NT also have NZ")

print("\n" + "="*80)
print("✅ ANALYSIS COMPLETE")
print("="*80)
print(f"\n📁 Output files saved to /mnt/user-data/outputs/")
print(f"   1. company_sbti_progressions.csv - Full progression data for all companies")

FileNotFoundError: [Errno 2] No such file or directory: 'historic new.xlsx'